#Scheduling Sports Equipment Production


**Author**: Rakshitha V S

**Date** : 10.02.2024

In [ ]:
# Install dependencies
!pip install -q amplpy ampltools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 29.9 MB/s eta 0:00:00


In [ ]:
# Google Colab & AMPL integration
MODULES, LICENSE_UUID = ["coin", 'gurobi', "cplex", "highs", "gokestrel"], "42fc7eb6-69aa-445d-b655-3ad24d836541"
from amplpy import tools
from ampltools import cloud_platform_name, ampl_notebook, register_magics

# instantiate AMPL object and register magics
if cloud_platform_name() is None:
    ampl = AMPL() # Use local installation of AMPL
else:
    ampl = tools.ampl_notebook(modules=MODULES, license_uuid=LICENSE_UUID, g=globals())

register_magics(ampl_object=ampl)

Licensed to Bundle #6741.7193 expiring 20241231: INFO 645 Prescriptive Analytics, Prof. Paul Brooks, Virginia Commonwealth University.


Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Read data

In [ ]:
import pandas as pd

# Load the data from the three sheets of the Excel file

path_to_data = "/content/drive/MyDrive/645 - PRESCRIPTIVE ANALYSIS/Brendamore.xlsx"
demand_data = pd.read_excel(path_to_data, sheet_name="Data", index_col=0)
details_data = pd.read_excel(path_to_data, sheet_name="Sheet1", index_col=0)
capacity_data = pd.read_excel(path_to_data, sheet_name="Sheet2", index_col=0)

# Define months and products
M = [1, 2, 3, 4, 5, 6]  # Month indices
Mplus0 = [0] + M
B = ["Footballs", "Soccer Balls"]

# Extract relevant columns for demand, production, and holding costs
demand = {
    "Footballs": dict(zip(M, demand_data["Football Demand Forecast"].tolist())),
    "Soccer Balls": dict(zip(M, demand_data["Soccer Ball Demand Forecast"].tolist()))
}

production = {
    "Footballs": dict(zip(M, demand_data["Production Cost ($ per football)"].tolist())),
    "Soccer Balls": dict(zip(M, demand_data["Production Cost ($ per soccer ball)"].tolist()))
}

holding = {
    "Footballs": dict(zip(M, demand_data["Holding Cost ($ per football)"].tolist())),
    "Soccer Balls": dict(zip(M, demand_data["Holding Cost ($ per soccer ball)"].tolist()))
}

# Extract initial and final inventory data
initial_inventory = dict(zip(B, details_data.loc["Current Inventory"].tolist()))
final_inventory = dict(zip(B, details_data.loc["Ending Inventory required (end of month 6)"].tolist()))

# Extract production and inventory capacity
production_capacity = capacity_data.loc["Production Capacity per Month", "Total"]
inventory_capacity = capacity_data.loc["Inventory Capacity", "Total"]

print(demand)
print(production)
print(holding)
print(initial_inventory)
print(final_inventory)
print(production_capacity)
print(inventory_capacity)

{'Footballs': {1: 15000, 2: 25000, 3: 20000, 4: 5000, 5: 2500, 6: 5000}, 'Soccer Balls': {1: 10000, 2: 15000, 3: 10000, 4: 5000, 5: 5000, 6: 7500}}
{'Footballs': {1: 13.8, 2: 13.9, 3: 12.95, 4: 12.6, 5: 12.55, 6: 12.7}, 'Soccer Balls': {1: 10.85, 2: 10.55, 3: 10.5, 4: 10.5, 5: 10.55, 6: 10.0}}
{'Footballs': {1: 0.6900000000000001, 2: 0.6950000000000001, 3: 0.6475, 4: 0.63, 5: 0.6275000000000001, 6: 0.635}, 'Soccer Balls': {1: 0.5425, 2: 0.5275000000000001, 3: 0.525, 4: 0.525, 5: 0.5275000000000001, 6: 0.5}}
{'Footballs': 7000, 'Soccer Balls': 5000}
{'Footballs': 3000, 'Soccer Balls': 3000}
32000
20000


Define model.

In [ ]:
# AMPL Model
ampl.eval ('''

reset;

set M;  # Months {1, 2, 3, 4, 5, 6}, indexed by i
set Mplus0;  # Includes period 0 for initial inventory {0, 1, 2, 3, 4, 5, 6}, indexed by i
set B;  # Products {footballs, soccer_balls}, indexed by j

# Decision Variables
var y {j in B, i in M} >= 0;  # Footballs and soccer balls produced each month, indexed by i and j
var x {j in B, i in Mplus0} >= 0;  # Inventory of footballs and soccer balls at the end of each month, indexed by i and j

# Parameters
param d {j in B, i in M};  # Demand for each product in each month, indexed by i and j
param p {j in B, i in M};  # Production cost for each product in each month, indexed by i and j
param h {j in B, i in M};  # Holding cost for each product in each month, indexed by i and j
param initial_inventory {j in B};  # Initial inventory for each product at period 0
param final_inventory {j in B};  # Final inventory required at the end of month 6
param production_capacity;  # Total production capacity per month (both products combined)
param inventory_capacity;  # Total storage capacity per month (both products combined)

# Objective: Minimize the total cost (production + holding)
minimize total_cost:
    sum {j in B, i in M} (p[j, i] * y[j, i] + h[j, i] * x[j, i]);

# Constraints:
# Inventory balance constraints for footballs and soccer balls
s.t. inventory_balance_constraint {j in B, i in M}:
    x[j, i-1] + y[j, i] - d[j, i] = x[j, i];

# Initial and final inventory constraints
s.t. initial_inventory_constraint {j in B}:
    x[j, 0] = initial_inventory[j];

s.t. final_inventory_constraint {j in B}:
    x[j, 6] >= final_inventory[j];

# Production capacity constraint (for each month)
s.t. production_capacity_constraint {i in M}:
    sum {j in B} y[j, i] <= production_capacity;

# Storage capacity constraint (for each month)
s.t. inventory_capacity_constraint {i in M}:
    sum {j in B} x[j, i] <= inventory_capacity;

''')


Define data and provide to AMPL model

In [ ]:
# Load data into AMPL
ampl.set['M'] = M
ampl.set['Mplus0'] = Mplus0
ampl.set['B'] = B

# Load parameters (demand, production cost, holding cost, inventory)
ampl.param['d'] = {(j, i): demand[j][i] for j in B for i in M}
ampl.param['p'] = {(j, i): production[j][i] for j in B for i in M}
ampl.param['h'] = {(j, i): holding[j][i] for j in B for i in M}
ampl.param['initial_inventory'] = initial_inventory
ampl.param['final_inventory'] = final_inventory
ampl.param['production_capacity'] = production_capacity
ampl.param['inventory_capacity'] = inventory_capacity

Use ampl.expand to confirm AMPL model syntax is working

In [ ]:
ampl.eval('''expand;''')

minimize total_cost:
	13.8*y['Footballs',1] + 13.9*y['Footballs',2] + 12.95*y['Footballs',3]
	 + 12.6*y['Footballs',4] + 12.55*y['Footballs',5] + 
	12.7*y['Footballs',6] + 10.85*y['Soccer Balls',1] + 
	10.55*y['Soccer Balls',2] + 10.5*y['Soccer Balls',3] + 
	10.5*y['Soccer Balls',4] + 10.55*y['Soccer Balls',5] + 
	10*y['Soccer Balls',6] + 0.69*x['Footballs',1] + 0.695*x['Footballs',2]
	 + 0.6475*x['Footballs',3] + 0.63*x['Footballs',4] + 
	0.6275*x['Footballs',5] + 0.635*x['Footballs',6] + 
	0.5425*x['Soccer Balls',1] + 0.5275*x['Soccer Balls',2] + 
	0.525*x['Soccer Balls',3] + 0.525*x['Soccer Balls',4] + 
	0.5275*x['Soccer Balls',5] + 0.5*x['Soccer Balls',6];

subject to inventory_balance_constraint[
	'Footballs',1]:
	y['Footballs',1] + x['Footballs',0] - x['Footballs',1] = 15000;

subject to inventory_balance_constraint[
	'Footballs',2]:
	y['Footballs',2] + x['Footballs',1] - x['Footballs',2] = 25000;

subject to inventory_balance_constraint[
	'Footballs',3]:
	y['Footballs',3] + x['F

Set solver option and solve

In [ ]:
ampl.setOption('solver', 'cbc')
ampl.solve()

cbc 2.10.10: cbc 2.10.10: optimal solution; objective 1448750
0 simplex iterations


Get and print results

In [ ]:
# Print the total cost
obj = ampl.getObjective('total_cost')
print("\nTotal cost is: ", obj.get().value(), "\n")

# Print production values
print("Production values:")
ampl.display('y')

# Print inventory values
print("Inventory values:")
ampl.display('x')



Total cost is:  1448750.0 

Production values:
y :=
Footballs      1   16000
Footballs      2   17000
Footballs      3   20000
Footballs      4    5000
Footballs      5    2500
Footballs      6    8000
'Soccer Balls' 1    5000
'Soccer Balls' 2   15000
'Soccer Balls' 3   10000
'Soccer Balls' 4    5000
'Soccer Balls' 5    5000
'Soccer Balls' 6   10500
;

Inventory values:
x :=
Footballs      0   7000
Footballs      1   8000
Footballs      2      0
Footballs      3      0
Footballs      4      0
Footballs      5      0
Footballs      6   3000
'Soccer Balls' 0   5000
'Soccer Balls' 1      0
'Soccer Balls' 2      0
'Soccer Balls' 3      0
'Soccer Balls' 4      0
'Soccer Balls' 5      0
'Soccer Balls' 6   3000
;



By the end of the six-month period, Brendamore Sports will have 3,000 footballs and 3,000 soccer balls in inventory, achieving an optimal total cost of \$1,448,750.